# 5. Walk-forward models

**Question this notebook answers**
Across many refit origins rather than one, does any model beat the random walk by more than the
spread of its own fold-to-fold variation? Answer is a mean and standard deviation per model, set
against the baselines from notebook 03.

**What v1 did**
`p1_models.ipynb` and `p2_models.ipynb` both use a single chronological split at 2018-12 and report
one number per model with no interval.

**Why that was wrong**
One split on this test window, at the persistence `00_audit_of_v1.ipynb` section 5 measures, leaves
only a handful of independent observations, so a single metric cannot be distinguished from noise. It also makes the
result an accident of where the split fell: the test window happens to hold about a fifth of the
training window's variance, which is why macro-only models produce such extreme negative
R-squareds. Reporting one number from one split invites exactly the over-reading that occurred.

**What this notebook does instead**
Runs roughly fifteen expanding-window origins with an embargo, refits every model at each origin,
and reports the distribution of each metric rather than a point estimate. Every model is scored
against the notebook 03 baselines on identical folds.

**Predicted effect on results — recorded before running**
I expect the fold-to-fold standard deviation of MSE to be large relative to the gaps between
models, so that the model ranking is not stable across origins. I expect no model to beat the
random walk on a majority of folds. Macro-only models should be clearly worse everywhere. If any
model does win consistently, the first thing to check is whether a feature has re-acquired a
lookahead.

**Inputs** → **Outputs**
`data/v2/panel_pit.parquet`, fold-wise regime labels from notebook 04 → per-fold rows appended to
`results/metrics.csv` via `src.evaluate.write_metrics`; figures to `results/figures/`.

---
## 5.1 Define the folds

Build folds with `src.splits.walk_forward_folds` and print `fold_summary`. Look for the embargo removing rows at the end of every training block and roughly fifteen origins.

---
## 5.2 Confirm no fold leaks

Call `src.splits.assert_no_overlap` on every fold explicitly, so the guarantee is tested rather than assumed.

---
## 5.3 Baselines on the same folds

Re-score the notebook 03 baselines fold by fold. Every model comparison below uses these identical folds.

---
## 5.4 Linear models

Fit a regularised linear model per fold. Refit any scaler inside the fold.

---
## 5.5 Tree ensembles

Fit a random forest and a gradient-boosted model per fold, with and without regime features.

---
## 5.6 With and without spread history

Run each model both with and without the spread's own lags. This is the comparison v1's Phase 2 got wrong by denying the sequence models spread history while giving it to the random forest.

---
## 5.7 Distribution of metrics across folds

Report mean and standard deviation per model via `src.evaluate.summarise_across_folds`. Look at whether the gaps between models exceed their own fold-to-fold spread.

---
## 5.8 Win rate against the random walk

Count the folds on which each model beats the random walk. A model that wins on half the folds has not demonstrated anything.

---
## 5.9 Write per-fold metrics

Call `src.evaluate.write_metrics` with every fold-model pair. This file is the only source notebook 06 reads.